In [22]:
import pandas as pd
import numpy as np
import sklearn

In [23]:
URL = "https://github.com/ageron/handson-ml2/blob/master/datasets/housing/housing.csv?raw=true"
df=pd.read_csv(URL)

In [24]:
from sklearn.model_selection import train_test_split
train_set, test_set = train_test_split(df, test_size=0.2, random_state=42)
X_train=train_set.drop("median_house_value", axis=1)
y=train_set['median_house_value'].copy()
X_num=X_train.drop("ocean_proximity", axis=1)

In [25]:
from sklearn.base import BaseEstimator, TransformerMixin
rooms_ix, bedrooms_ix, population_ix, households_ix = 3, 4, 5, 6
class CombinedAttributesAdder(BaseEstimator, TransformerMixin):
    def __init__(self, add_bedrooms_per_room = True):
        self.add_bedrooms_per_room = add_bedrooms_per_room
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        rooms_per_household = X[:, rooms_ix] / X[:, households_ix]
        population_per_household = X[:, population_ix] / X[:, households_ix]
        if self.add_bedrooms_per_room:
            bedrooms_per_room=X[:, bedrooms_ix] / X[:, rooms_ix]
            return np.c_[X, rooms_per_household, population_per_household, bedrooms_per_room]
        else:
            return np.c_[X, rooms_per_household, population_per_household]

In [26]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

num_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy="median")),
        ('attribs_adder', CombinedAttributesAdder()),
        ('std_scaler', StandardScaler()),
    ])


In [27]:
from sklearn.compose import ColumnTransformer
num_attribs = list(X_num)
cat_attribs = ["ocean_proximity"]

full_pipeline = ColumnTransformer([
        ("num", num_pipeline, num_attribs),
        ("cat", OneHotEncoder(), cat_attribs)
])


In [28]:
X_prepared = full_pipeline.fit_transform(X_train)

In [29]:
from sklearn.linear_model import LinearRegression
lin_model = LinearRegression()
lin_model.fit(X_prepared, y)

LinearRegression()

In [30]:
test_data = X_train.sample(10)
test_labels = y.loc[test_data.index]
test_data

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity
17001,-122.26,37.57,23.0,7995.0,1254.0,3484.0,1198.0,6.5948,NEAR BAY
2679,-115.53,32.99,25.0,2578.0,634.0,2082.0,565.0,1.7159,INLAND
12840,-121.43,38.66,35.0,1814.0,367.0,1076.0,372.0,2.7179,INLAND
4093,-118.42,34.16,46.0,54.0,9.0,20.0,6.0,0.5360,<1H OCEAN
2954,-119.01,35.35,39.0,598.0,149.0,366.0,132.0,1.9125,INLAND
18538,-122.01,36.98,27.0,2820.0,730.0,1511.0,745.0,2.5890,NEAR OCEAN
20482,-118.72,34.28,17.0,2654.0,478.0,1392.0,451.0,5.4459,<1H OCEAN
15924,-122.44,37.72,52.0,1775.0,347.0,1102.0,367.0,4.3125,NEAR BAY
7743,-118.15,33.93,25.0,1948.0,433.0,1128.0,429.0,3.7614,<1H OCEAN
3724,-118.42,34.18,27.0,3760.0,880.0,2022.0,812.0,3.1551,<1H OCEAN


In [31]:
tets_label=y.loc[test_data.index]
tets_label

,median_house_value
17001,404000.0
2679,62200.0
12840,81100.0
4093,375000.0
2954,57900.0
18538,242400.0
20482,223900.0
15924,267200.0
7743,255900.0
3724,225600.0


In [37]:
test_data_prepared = full_pipeline.transform(test_data)


In [34]:
predict_labels=lin_model.predict(test_data_prepared)
predict_labels

array([344435.38498599,  34392.99645588, 105836.42847647,  95600.27875971,
       106767.37251569, 232097.27775385, 269695.98851286, 269683.44656914,
       215957.56313384, 212307.40189018])

In [36]:
test_labels

,median_house_value
17001,404000.0
2679,62200.0
12840,81100.0
4093,375000.0
2954,57900.0
18538,242400.0
20482,223900.0
15924,267200.0
7743,255900.0
3724,225600.0


In [38]:
pd.DataFrame({'Actual': test_labels, 'Predicted': predict_labels})

,Actual,Predicted
17001,404000.0,344435.384986
2679,62200.0,34392.996456
12840,81100.0,105836.428476
4093,375000.0,95600.278760
2954,57900.0,106767.372516
18538,242400.0,232097.277754
20482,223900.0,269695.988513
15924,267200.0,269683.446569
7743,255900.0,215957.563134
3724,225600.0,212307.401890


In [39]:
# 5 - qadam modelni baholash
test_set

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
20046,-119.01,36.06,25.0,1505.0,NaN,1392.0,359.0,1.6812,47700.0,INLAND
3024,-119.46,35.14,30.0,2943.0,NaN,1565.0,584.0,2.5313,45800.0,INLAND
15663,-122.44,37.80,52.0,3830.0,NaN,1310.0,963.0,3.4801,500001.0,NEAR BAY
20484,-118.72,34.28,17.0,3051.0,NaN,1705.0,495.0,5.7376,218600.0,<1H OCEAN
9814,-121.93,36.62,34.0,2351.0,NaN,1063.0,428.0,3.7250,278000.0,NEAR OCEAN
...,...,...,...,...,...,...,...,...,...,...
15362,-117.22,33.36,16.0,3165.0,482.0,1351.0,452.0,4.6050,263300.0,<1H OCEAN
16623,-120.83,35.36,28.0,4323.0,886.0,1650.0,705.0,2.7266,266800.0,NEAR OCEAN
18086,-122.05,37.31,25.0,4111.0,538.0,1585.0,568.0,9.2298,500001.0,<1H OCEAN
2144,-119.76,36.77,36.0,2507.0,466.0,1227.0,474.0,2.7850,72300.0,INLAND


In [40]:
X_test = test_set.drop('median_house_value', axis=1)
X_test

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity
20046,-119.01,36.06,25.0,1505.0,NaN,1392.0,359.0,1.6812,INLAND
3024,-119.46,35.14,30.0,2943.0,NaN,1565.0,584.0,2.5313,INLAND
15663,-122.44,37.80,52.0,3830.0,NaN,1310.0,963.0,3.4801,NEAR BAY
20484,-118.72,34.28,17.0,3051.0,NaN,1705.0,495.0,5.7376,<1H OCEAN
9814,-121.93,36.62,34.0,2351.0,NaN,1063.0,428.0,3.7250,NEAR OCEAN
...,...,...,...,...,...,...,...,...,...
15362,-117.22,33.36,16.0,3165.0,482.0,1351.0,452.0,4.6050,<1H OCEAN
16623,-120.83,35.36,28.0,4323.0,886.0,1650.0,705.0,2.7266,NEAR OCEAN
18086,-122.05,37.31,25.0,4111.0,538.0,1585.0,568.0,9.2298,<1H OCEAN
2144,-119.76,36.77,36.0,2507.0,466.0,1227.0,474.0,2.7850,INLAND


In [41]:
y_test=test_set['median_house_value'].copy()
y_test

,median_house_value
20046,47700.0
3024,45800.0
15663,500001.0
20484,218600.0
9814,278000.0
...,...
15362,263300.0
16623,266800.0
18086,500001.0
2144,72300.0


In [42]:
X_test_prepared = full_pipeline.transform(X_test)

In [43]:
y_predict = lin_model.predict(X_test_prepared)


In [44]:
y_predict

array([ 61874.25460143, 121853.52511139, 267770.94368091, ...,
       447837.04647878, 117275.9214608 , 185597.46125194])

In [45]:
#o'rtacha xatolik
from sklearn.metrics import mean_absolute_error
mae = mean_absolute_error(y_test, y_predict)
mae


50898.73953494079

In [50]:
# o'rtacha kvadratik xatolik
from sklearn.metrics import mean_squared_error
rmse = np.sqrt(mean_squared_error(y_test, y_predict))
rmse

np.float64(72701.32600762135)

In [51]:
# random forest
from sklearn.ensemble import RandomForestRegressor
RF_model = RandomForestRegressor()
RF_model.fit(X_prepared, y)

RandomForestRegressor()

In [55]:
y_predict = RF_model.predict(X_test_prepared)


In [56]:
from sklearn.metrics import mean_squared_error
rmse = np.sqrt(mean_squared_error(y_test, y_predict))
rmse

np.float64(50738.987588848155)

In [57]:
#CrossValidation - bir nechta qismga bo'lib qayta qayta ishga tushirilganda har gal bo'lgan qismini test uchun olib qo'yib qolgan qismini train qiladi
x=df.drop("median_house_value", axis=1)
y=df['median_house_value'].copy()
X_prepared = full_pipeline.fit_transform(x)

In [60]:
from sklearn.model_selection import cross_val_score
mse_scores = cross_val_score(lin_model, X_prepared, y, scoring="neg_mean_squared_error", cv=5)

In [61]:
def display_scores(scores):
    print("Scores:", scores)
    print("Mean:", scores.mean())
    print("Standard deviation:", scores.std())

In [62]:
display_scores(np.sqrt(-mse_scores))

Scores: [73391.42036892 74809.28332317 75429.91837496 76604.35506436
 66196.72436926]
Mean: 73286.34030013601
Standard deviation: 3693.161254481324


In [63]:
scores = cross_val_score(RF_model, X_prepared, y, scoring="neg_mean_squared_error", cv=10)
LR_rmse_scores = np.sqrt(-scores)
display_scores(LR_rmse_scores)

Scores: [97125.96594829 47915.41905488 65160.62640531 56342.38478162
 60824.57736862 59881.75103469 47077.63373498 78523.65146656
 74532.55367653 49244.68462803]
Mean: 63662.92480994786
Standard deviation: 15059.526970886045


In [64]:
#pickle faylni saqlab olish

In [65]:
import pickle
filename = 'RF_model.pkl'
with open(filename, 'wb') as file:
  pickle.dump(RF_model, file)

In [67]:
#joblib
import joblib
filename = 'lin_model.jbl'
joblib.dump(lin_model, filename)

['lin_model.jbl']

In [70]:
model = joblib.load(filename)


In [71]:
scores = cross_val_score(RF_model, X_prepared, y, scoring="neg_mean_squared_error", cv=10)
LR_rmse_scores = np.sqrt(-scores)
display_scores(LR_rmse_scores)

Scores: [97601.87650914 47425.05833849 65262.06552262 56362.79988152
 60728.85681972 60265.16252699 46923.54913385 80353.78710457
 74803.14217826 49553.49085236]
Mean: 63927.978886752
Standard deviation: 15403.872836469289
